In [ ]:
"""Temporary notebook to visually inspect preprocessed test patches.

What it does:
- Loads original and preprocessed (normalized) PCam test patches.
- Computes per-patch distance to the stain reference before/after.
- Shows:
  * Test patches whose distance to the reference increased the most after normalization.
  * Patches flagged as "purple-only" after normalization.
- For both sets, shows original vs normalized side by side for human inspection.

Assumptions:
- Project root: c:\GP_ECG
- Original PCam data under: pcam_data/test/camelyonpatch_level_2_split_test_x.h5
- Preprocessed data under: pcam_data/preprocessed_multi_ref/test_x.h5
- Preprocessing report: pcam_data/preprocessed_multi_ref/preprocess_report.json
"""

import os
import json

import h5py
import numpy as np
import matplotlib.pyplot as plt

# --- Configuration ---
PROJECT_ROOT = r"c:\GP_ECG"
DATA_DIR = os.path.join(PROJECT_ROOT, "pcam_data")
PREPROC_DIR = os.path.join(DATA_DIR, "preprocessed_multi_ref")  # use "preprocessed" for older single-ref run

orig_test_x_path = os.path.join(DATA_DIR, "test", "camelyonpatch_level_2_split_test_x.h5")
norm_test_x_path = os.path.join(PREPROC_DIR, "test_x.h5")
report_path = os.path.join(PREPROC_DIR, "preprocess_report.json")

print("Original test X:", orig_test_x_path)
print("Preprocessed test X:", norm_test_x_path)
print("Report:", report_path)

# --- Load config / reference ---
with open(report_path, "r") as f:
    report = json.load(f)
ref_mean_rgb = np.array(report["config"]["ref_mean_rgb"], dtype=np.float32)
print("Reference mean RGB (from preprocessing):", ref_mean_rgb)

# --- Load data ---
with h5py.File(orig_test_x_path, "r") as f_orig, h5py.File(norm_test_x_path, "r") as f_norm:
    x_orig = f_orig["x"][:]   # uint8 [0,255], full original test set
    x_norm = f_norm["x"][:]   # float32 [0,1], only kept (quality-passed) test patches

print("Original test shape:", x_orig.shape)
print("Normalized test shape:", x_norm.shape)

# Load mapping from preprocessed position -> original index
manifest_path = os.path.join(PREPROC_DIR, "manifest.json")
with open(manifest_path, "r") as f:
    manifest = json.load(f)
kept_test = np.array(manifest["test"]["kept_indices"], dtype=np.int64)
assert kept_test.shape[0] == x_norm.shape[0], "kept_indices and normalized test_x length mismatch"

# Convert original to [0,1] to match normalized scale
x_orig_01 = x_orig.astype(np.float32) / 255.0

# --- Compute per-patch metrics on a random sample of *kept* test patches ---

def mean_rgb_and_pink(x01: np.ndarray):
    """Return (mean_rgb, pink_pct) for an image in [0,1], shape (H,W,3)."""
    flat = x01.reshape(-1, 3)
    mean_rgb = flat.mean(axis=0)
    r = x01[..., 0]
    b = x01[..., 2]
    pink_pct = (r > b).mean()
    return mean_rgb, pink_pct

n_total = x_norm.shape[0]  # only kept/normalized test patches
print("Total KEPT test patches (after quality filter):", n_total)

rng = np.random.RandomState(42)
sample_size = min(5000, n_total)
# work in normalized index space 0..n_total-1
sample_pos = rng.choice(n_total, size=sample_size, replace=False)
print("Sampling", sample_size, "kept test patches for analysis.")

mean_ref = ref_mean_rgb

dist_before = []
dist_after = []
is_purple_only = []

for pos in sample_pos:
    orig_idx = kept_test[pos]          # index into original test_x
    m_bef, _ = mean_rgb_and_pink(x_orig_01[orig_idx])
    m_aft, pink_aft = mean_rgb_and_pink(x_norm[pos])

    d_bef = np.abs(m_bef - mean_ref).sum()
    d_aft = np.abs(m_aft - mean_ref).sum()

    dist_before.append(d_bef)
    dist_after.append(d_aft)

    # "Purple-only" criterion: very low red mean OR very low pink fraction
    purple_flag = (m_aft[0] < 0.2) or (pink_aft < 0.05)
    is_purple_only.append(purple_flag)

# Convert to arrays

dist_before = np.array(dist_before)
dist_after = np.array(dist_after)
is_purple_only = np.array(is_purple_only)

delta = dist_after - dist_before  # >0 => distance increased (worse)

print("Mean distance before / after (sample):", float(dist_before.mean()), "/", float(dist_after.mean()))
print("Fraction purple-only after norm (sample):", float(is_purple_only.mean()))

# Keep track of which normalized positions we sampled, for plotting later
sample_pos = np.array(sample_pos, dtype=np.int64)

# --- Helper to show original vs normalized side by side ---

import math

def show_before_after(indices, title, n_pairs_per_row: int = 2):
    """Show original vs normalized for given global indices (test split).

    Layout: each row has n_pairs_per_row examples; each example uses 2 columns (orig, norm).
    """
    if len(indices) == 0:
        print("No indices to show for:", title)
        return

    n = len(indices)
    n_cols = n_pairs_per_row * 2
    n_rows = math.ceil(n / n_pairs_per_row)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(3 * n_cols, 3 * n_rows))
    axes = np.atleast_1d(axes).ravel()

    ax_i = 0
    for global_idx in indices:
        orig = x_orig_01[global_idx]
        norm = x_norm[global_idx]

        # Original
        ax = axes[ax_i]
        ax.imshow(orig)
        ax.set_title(f"orig idx={global_idx}")
        ax.axis("off")
        ax_i += 1

        # Normalized
        ax = axes[ax_i]
        ax.imshow(norm)
        ax.set_title(f"norm idx={global_idx}")
        ax.axis("off")
        ax_i += 1

    for j in range(ax_i, len(axes)):
        axes[j].axis("off")

    fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()

# --- 1) Show test patches whose distance increased the most ---

k = 48  # number of examples to inspect (grid rows); lower if slow
worst_idx_order = np.argsort(delta)[::-1]
worst_pos = sample_pos[worst_idx_order[:k]]  # positions in normalized test_x

print("Showing", len(worst_pos), "test patches with largest increase in distance to reference.")
show_before_after(worst_pos, "Test patches with largest increase in distance to reference (orig vs norm)")

# --- 2) Show purple-only patches after normalization ---

purple_pos = sample_pos[is_purple_only]
print(f"Purple-only after norm in sample: {len(purple_pos)} / {len(sample_pos)}")

show_before_after(purple_pos[:k], "Test patches flagged as purple-only after normalization (orig vs norm)")
